In [19]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

In [5]:
# 1. Leer el archivo de ancho fijo (FWF) omitiendo líneas en blanco
df = pd.read_fwf("estaciones_smn.txt", encoding="latin-1").dropna(how="all")

# Limpiar posibles espacios extra en los nombres de las columnas
df.columns = df.columns.str.strip()

# 2. Filtrar por las provincias de la región NEA
provincias_nea = ["MISIONES", "CORRIENTES", "FORMOSA", "CHACO"]
df_nea = df[df["PROVINCIA"].str.strip().isin(provincias_nea)].copy()

# Convertir la columna ALTURA a tipo numérico por seguridad
df_nea["ALTURA"] = pd.to_numeric(df_nea["ALTURA"], errors="coerce")

# -------------------------------------------------------------
# RESULTADOS PARA TUS PREGUNTAS:
# -------------------------------------------------------------
cant_estaciones = df_nea["NOMBRE"].nunique()
altura_media = df_nea["ALTURA"].mean()

print(f"1. Cantidad de estaciones en NEA: {cant_estaciones}")
print(f"2. Altura media: {altura_media:.2f} m")

1. Cantidad de estaciones en NEA: 13
2. Altura media: 170.23 m


In [12]:
# 1. Cargar el dataset especificando el separador ';'
# Omitimos las líneas de texto si hubiera encabezados raros o usamos nombres de columnas claros
df_obs = pd.read_csv(
    "estado_tiempo20260905.txt",
    sep=";",
    encoding="latin-1",
    header=None,
    usecols=[0, 1, 2, 3, 4, 5],
    names=["estacion", "fecha", "hora", "estado", "visibilidad", "temperatura"],
)

# Limpiar espacios en blanco de las columnas de texto
df_obs["estacion"] = df_obs["estacion"].str.strip()

# 2. Lista de estaciones pertenecientes a la región NEA (Chaco, Corrientes, Formosa, Misiones)
estaciones_nea = [
    "Pcia. Roque Saenz Peña",
    "Resistencia",
    "Corrientes",
    "Ituzaingó",
    "Mercedes",
    "Monte Caseros",
    "Paso De Los Libres",
    "Formosa",
    "Las Lomitas",
    "Bernardo De Irigoyen",
    "Iguazú",
    "Oberá",
    "Posadas",
]

# 3. Filtrar el dataframe por las estaciones del NEA
df_nea_obs = df_obs[df_obs["estacion"].isin(estaciones_nea)].copy()

# Asegurar que la temperatura sea numérica
df_nea_obs["temperatura"] = pd.to_numeric(
    df_nea_obs["temperatura"], errors="coerce"
)

# 4. Calcular la temperatura media
temp_media = df_nea_obs["temperatura"].mean()

print(f"Temperaturas registradas:\n{df_nea_obs[['estacion', 'temperatura']]}")
print(f"\nTemperatura media de la región NEA: {temp_media:.2f} °C")

# Filtrar y calcular desviación estándar
df_nea = df_obs[df_obs["estacion"].str.strip().isin(estaciones_nea)].copy()
df_nea["temperatura"] = pd.to_numeric(df_nea["temperatura"], errors="coerce")

desvio = df_nea["temperatura"].std()
print(f"Desvío estándar de la temperatura en el NEA: {desvio:.2f} °C")

Temperaturas registradas:
                  estacion  temperatura
30  Pcia. Roque Saenz Peña         21.7
31             Resistencia         21.7
46              Corrientes         21.0
47               Ituzaingó         19.8
48                Mercedes         19.0
49           Monte Caseros         18.8
50      Paso De Los Libres         19.5
54                 Formosa         22.1
55             Las Lomitas         18.4
73    Bernardo De Irigoyen         15.3
74                  Iguazú         18.2
75                   Oberá         14.8
76                 Posadas         20.8

Temperatura media de la región NEA: 19.32 °C
Desvío estándar de la temperatura en el NEA: 2.30 °C


In [14]:
file_path = "Estadísticas normales Datos abiertos 1991-2020.txt"
meses = [
    "Ene",
    "Feb",
    "Mar",
    "Abr",
    "May",
    "Jun",
    "Jul",
    "Ago",
    "Sep",
    "Oct",
    "Nov",
    "Dic",
]

registros = []

# 1. Leer el archivo y obtener filas de Precipitación para Córdoba
with open(file_path, "r", encoding="latin1") as f:
    for line in f:
        if "CÓRDOBA" in line and "Precipitación (mm)" in line:
            partes = [p.strip() for p in line.strip().split("\t") if p.strip()]
            estacion = partes[0]
            variable = partes[1]
            valores = [float(v) for v in partes[2:14]]
            registros.append([estacion, variable] + valores)

# 2. Crear el DataFrame con tipos numéricos
df_precip = pd.DataFrame(registros, columns=["Estación", "Variable"] + meses)
df_precip[meses] = df_precip[meses].apply(pd.to_numeric)

# 3. Calcular e identificar el mes con mayor precipitación por estación
for idx, row in df_precip.iterrows():
    valores_meses = row[meses]
    mes_max = valores_meses.idxmax()
    val_max = valores_meses.max()
    print(
        f"Estación: {row['Estación']} -> Mes con mayor precipitación: {mes_max} ({val_max} mm)"
    )

Estación: CÓRDOBA AERO -> Mes con mayor precipitación: Dic (146.5 mm)
Estación: CÓRDOBA OBSERVATORIO -> Mes con mayor precipitación: Dic (143.9 mm)


In [15]:
file_path = "Estadísticas normales Datos abiertos 1991-2020.txt"

# Mapeo de estaciones del NEA a sus respectivas provincias
estacion_provincia = {
    "FORMOSA AERO": "FORMOSA",
    "RESISTENCIA AERO": "CHACO",
    "PRESIDENCIA ROQUE SAENZ PEÑA AERO*": "CHACO",
    "POSADAS AERO": "MISIONES",
    "IGUAZÚ AERO": "MISIONES",
    "BERNARDO DE IRIGOYEN AERO": "MISIONES",
    "CORRIENTES AERO": "CORRIENTES",
    "PASO DE LOS LIBRES AERO": "CORRIENTES",
    "MONTE CASEROS AERO": "CORRIENTES",
}

registros = []

# 1. Leer el archivo y extraer valores de Temperatura (°C)
with open(file_path, "r", encoding="latin1") as f:
    for line in f:
        partes = [p.strip() for p in line.strip().split("\t") if p.strip()]

        if len(partes) >= 14 and partes[1] == "Temperatura (°C)":
            estacion = partes[0]
            if estacion in estacion_provincia:
                try:
                    valores_mensuales = [float(x) for x in partes[2:14]]
                    temp_media_anual = sum(valores_mensuales) / len(
                        valores_mensuales
                    )
                    registros.append(
                        {
                            "Provincia": estacion_provincia[estacion],
                            "Estación": estacion,
                            "Temp_Media_Anual": round(temp_media_anual, 2),
                        }
                    )
                except ValueError:
                    # Omitir en caso de valores no disponibles (ej. 'S/D')
                    pass

# 2. Crear DataFrame
df = pd.DataFrame(registros)

# 3. Promediar por provincia
df_provincia = (
    df.groupby("Provincia")["Temp_Media_Anual"]
    .mean()
    .round(2)
    .reset_index()
    .sort_values(by="Temp_Media_Anual", ascending=False)
)

# 4. Obtener la provincia con mayor temperatura media
provincia_max = df_provincia.iloc[0]

print(df_provincia.to_string(index=False))
print(
    f"\nLa provincia con mayor temperatura media es {provincia_max['Provincia']} con {provincia_max['Temp_Media_Anual']} °C."
)

 Provincia  Temp_Media_Anual
   FORMOSA             22.45
     CHACO             21.30
  MISIONES             20.90
CORRIENTES             20.36

La provincia con mayor temperatura media es FORMOSA con 22.45 °C.


In [18]:
file_path = "Estadísticas normales Datos abiertos 1991-2020.txt"
meses = [
    "Ene",
    "Feb",
    "Mar",
    "Abr",
    "May",
    "Jun",
    "Jul",
    "Ago",
    "Sep",
    "Oct",
    "Nov",
    "Dic",
]

estaciones = [
    "FORMOSA AERO",
    "RESISTENCIA AERO",
    "PRESIDENCIA ROQUE SAENZ PEÑA AERO*",
    "POSADAS AERO",
    "IGUAZÚ AERO",
    "BERNARDO DE IRIGOYEN AERO",
    "CORRIENTES AERO",
    "PASO DE LOS LIBRES AERO",
    "MONTE CASEROS AERO",
]

data_p, data_t = [], []

with open(file_path, "r", encoding="latin1") as f:
    for line in f:
        partes = [p.strip() for p in line.strip().split("\t") if p.strip()]
        if len(partes) >= 14 and partes[0] in estaciones:
            if partes[1] == "Precipitación (mm)":
                data_p.append([partes[0]] + [float(x) for x in partes[2:14]])
            elif partes[1] == "Temperatura (°C)":
                data_t.append([partes[0]] + [float(x) for x in partes[2:14]])

df_p = pd.DataFrame(data_p, columns=["Estación"] + meses)
df_t = pd.DataFrame(data_t, columns=["Estación"] + meses)


def buscar_outliers_por_mes(df, variable_nombre):
    print(f"=== OUTLIERS EN {variable_nombre.upper()} (EVALUACIÓN POR MES) ===")
    hay_outliers = False
    for m in meses:
        q1 = df[m].quantile(0.25)
        q3 = df[m].quantile(0.75)
        iqr = q3 - q1
        outliers = df[(df[m] < q1 - 1.5 * iqr) | (df[m] > q3 + 1.5 * iqr)]
        if not outliers.empty:
            hay_outliers = True
            for _, row in outliers.iterrows():
                print(f"Mes {m}: {row['Estación']} -> {row[m]}")
    if not hay_outliers:
        print("No se detectaron outliers.")
    print()


buscar_outliers_por_mes(df_p, "Precipitación")
buscar_outliers_por_mes(df_t, "Temperatura")

=== OUTLIERS EN PRECIPITACIÓN (EVALUACIÓN POR MES) ===
Mes Ene: BERNARDO DE IRIGOYEN AERO -> 207.1
Mes Feb: BERNARDO DE IRIGOYEN AERO -> 189.1
Mes Abr: PRESIDENCIA ROQUE SAENZ PEÑA AERO* -> 117.6

=== OUTLIERS EN TEMPERATURA (EVALUACIÓN POR MES) ===
Mes Ene: BERNARDO DE IRIGOYEN AERO -> 22.9
Mes Feb: BERNARDO DE IRIGOYEN AERO -> 22.6
Mes Mar: BERNARDO DE IRIGOYEN AERO -> 21.7
Mes Ago: MONTE CASEROS AERO -> 15.1
Mes Dic: BERNARDO DE IRIGOYEN AERO -> 22.5



In [20]:
# Diccionario de estaciones con sus respectivas altitudes (msnm)
# (Puedes adaptar las altitudes y estaciones según la región elegida)
estaciones_altitud = {
    "FORMOSA AERO": 60,
    "RESISTENCIA AERO": 52,
    "PRESIDENCIA ROQUE SAENZ PEÑA AERO*": 90,
    "POSADAS AERO": 131,
    "IGUAZÚ AERO": 270,
    "BERNARDO DE IRIGOYEN AERO": 825,
    "CORRIENTES AERO": 62,
    "PASO DE LOS LIBRES AERO": 70,
    "MONTE CASEROS AERO": 54,
}

altitudes = []
temperaturas_medias = []

# 1. Leer el archivo y calcular la temperatura media anual por estación
with open(file_path, "r", encoding="latin1") as f:
    for line in f:
        partes = [p.strip() for p in line.strip().split("\t") if p.strip()]

        if len(partes) >= 14 and partes[1] == "Temperatura (°C)":
            estacion = partes[0]
            if estacion in estaciones_altitud:
                valores_mensuales = [float(x) for x in partes[2:14]]
                temp_anual = sum(valores_mensuales) / len(valores_mensuales)

                temperaturas_medias.append(temp_anual)
                altitudes.append(estaciones_altitud[estacion])

# 2. Calcular el coeficiente de correlación de Spearman
coeficiente, p_valor = spearmanr(altitudes, temperaturas_medias)

print(f"Coeficiente de Spearman: {coeficiente:.4f}")
print(f"P-valor: {p_valor:.4f}")

if coeficiente < 0:
    print(
        "Conclusión: Existe una relación inversa (a mayor altitud, menor temperatura)."
    )
elif coeficiente > 0:
    print("Conclusión: Existe una relación directa.")
else:
    print("Conclusión: No hay relación aparente.")

Coeficiente de Spearman: -0.0667
P-valor: 0.8647
Conclusión: Existe una relación inversa (a mayor altitud, menor temperatura).


In [21]:
# Mapeo de estaciones a provincias para la región (NEA)
estacion_provincia = {
    "FORMOSA AERO": "FORMOSA",
    "RESISTENCIA AERO": "CHACO",
    "PRESIDENCIA ROQUE SAENZ PEÑA AERO*": "CHACO",
    "POSADAS AERO": "MISIONES",
    "IGUAZÚ AERO": "MISIONES",
    "BERNARDO DE IRIGOYEN AERO": "MISIONES",
    "CORRIENTES AERO": "CORRIENTES",
    "PASO DE LOS LIBRES AERO": "CORRIENTES",
    "MONTE CASEROS AERO": "CORRIENTES",
}

data = []

# 1. Leer archivo y estructurar datos de precipitación
with open(file_path, "r", encoding="latin1") as f:
    for line in f:
        partes = [p.strip() for p in line.strip().split("\t") if p.strip()]
        if len(partes) >= 14 and partes[1] == "Precipitación (mm)":
            estacion = partes[0]
            if estacion in estacion_provincia:
                provincia = estacion_provincia[estacion]
                valores = [float(x) for x in partes[2:14]]
                for val in valores:
                    data.append({"Provincia": provincia, "Precipitación": val})

df = pd.DataFrame(data)

# 2. Cálculo de Eta Cuadrado (Suma de cuadrados inter-grupo / Suma de cuadrados total)
media_global = df["Precipitación"].mean()
ss_total = np.sum((df["Precipitación"] - media_global) ** 2)

ss_entre_grupos = 0
for provincia, grupo in df.groupby("Provincia"):
    n_k = len(grupo)
    media_k = grupo["Precipitación"].mean()
    ss_entre_grupos += n_k * ((media_k - media_global) ** 2)

eta_cuadrado = ss_entre_grupos / ss_total

print(f"Valor de Eta Cuadrado (η²): {eta_cuadrado:.4f}")

# 3. Interpretación según el resultado
if eta_cuadrado < 0.06:
    print("Conclusión: La asociación es débil.")
elif eta_cuadrado < 0.25:
    print(
        "Conclusión: La asociación es débil o moderada; hay poca relación entre las precipitaciones y las provincias."
    )
else:
    print(
        "Conclusión: Existe una asociación fuerte entre la provincia y la precipitación."
    )

Valor de Eta Cuadrado (η²): 0.2321
Conclusión: La asociación es débil o moderada; hay poca relación entre las precipitaciones y las provincias.


In [22]:


file_path = "Estadísticas normales Datos abiertos 1991-2020.txt"
meses = [
    "Ene",
    "Feb",
    "Mar",
    "Abr",
    "May",
    "Jun",
    "Jul",
    "Ago",
    "Sep",
    "Oct",
    "Nov",
    "Dic",
]

# Estación de la provincia elegida
estacion_objetivo = "FORMOSA AERO"
provincia = "Formosa"

datos_provincia = {}

# 1. Extraer las variables del archivo para Formosa Aero
with open(file_path, "r", encoding="latin1") as f:
    for line in f:
        partes = [p.strip() for p in line.strip().split("\t") if p.strip()]
        if len(partes) >= 14 and partes[0] == estacion_objetivo:
            variable = partes[1]
            valores = [float(x) for x in partes[2:14]]

            media_anual = sum(valores) / len(valores)
            idx_max = valores.index(max(valores))
            idx_min = valores.index(min(valores))

            datos_provincia[variable] = {
                "media_anual": round(media_anual, 1),
                "total_anual": round(sum(valores), 1),
                "max": (meses[idx_max], valores[idx_max]),
                "min": (meses[idx_min], valores[idx_min]),
            }

# 2. Asignar variables clave
temp = datos_provincia.get("Temperatura (°C)")
precip = datos_provincia.get("Precipitación (mm)")
humedad = datos_provincia.get("Humedad relativa (%)")
nubosidad = datos_provincia.get("Nubosidad total (octavos)")
viento = datos_provincia.get(
    "Velocidad del Viento (km/h) (2011-2020)"
)

# 3. Generar la redacción automática (más de 50 palabras)
texto_caracterizacion = f"""
Provincia elegida: {provincia}

La provincia de {provincia} presenta un clima subtropical cálido con las temperaturas medias más elevadas de la región, registrando un promedio anual de {temp['media_anual']} °C que oscila entre máximas estivales promedio de {temp['max'][1]} °C en {temp['max'][0]} y mínimas invernales de {temp['min'][1]} °C en {temp['min'][0]}.

En cuanto a la precipitación, la provincia registra un acumulado anual abundante de {precip['total_anual']} mm, concentrando sus mayores aportes hídricos durante la estación cálida (con un máximo en {precip['max'][0]} de {precip['max'][1]} mm) y reduciéndose progresivamente en invierno (con un mínimo en {precip['min'][0]} de {precip['min'][1]} mm).

Finalmente, la humedad relativa se mantiene elevada durante todo el año con un promedio constante del {humedad['media_anual']}%, alcanzando su pico en {humedad['max'][0]} ({humedad['max'][1]}%). Estas condiciones, combinadas con una nubosidad media de {nubosidad['media_anual']} octavos y vientos moderados de {viento['media_anual']} km/h, configuran un ambiente predominantemente cálido y húmedo característico del Gran Chaco.
"""

print(texto_caracterizacion)


Provincia elegida: Formosa

La provincia de Formosa presenta un clima subtropical cálido con las temperaturas medias más elevadas de la región, registrando un promedio anual de 22.4 °C que oscila entre máximas estivales promedio de 27.7 °C en Ene y mínimas invernales de 16.4 °C en Jul.

En cuanto a la precipitación, la provincia registra un acumulado anual abundante de 1406.5 mm, concentrando sus mayores aportes hídricos durante la estación cálida (con un máximo en Dic de 178.4 mm) y reduciéndose progresivamente en invierno (con un mínimo en Ago de 34.8 mm).

Finalmente, la humedad relativa se mantiene elevada durante todo el año con un promedio constante del 74.5%, alcanzando su pico en May (81.2%). Estas condiciones, combinadas con una nubosidad media de 3.9 octavos y vientos moderados de 12.7 km/h, configuran un ambiente predominantemente cálido y húmedo característico del Gran Chaco.

